# **LIMPIEZA Y TRANSFORMACIÓN DE LOS DATOS**

Este notebook aplica las transformaciones de limpieza derivadas del EDA y realiza validaciones básicas para asegurar consistencia y calidad de los datos antes del análisis.

**Principios**
- Transformar y exportar el dataset limpio a `data/procesada/`.
- Documentar cada transformación y su motivación.

In [70]:
import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

#### **CARGA DE LOS DATASETS**

- Importar los datasets originales que servirán como base para las fases de limpieza, corrección y transformación de los datos.
- Tras la carga, se realiza una verificación básica para confirmar la correcta lectura de los archivos, dimensiones y la estructura general.

##### **Descripción de los datasets**

Se cargan los dataset:

- **Customer online delivery dataset - Customer_data**  

Dataset que contiene datos de usuarios que utilizan y consumen aplicaciones de comida rápida, creando un pérfil sociodemográfico y de uso.

- **food_delivery_dataset** 

Dataset basado en operaciones, pedidos, tiempo estiamdo de llegada del pedido, costes. 


In [71]:
path_df1= "../Data/sin procesar/Customer online delivery dataset - Customer_data.csv"
path_df2= "../Data/sin procesar/food_delivery_dataset.csv"

df1 = pd.read_csv(path_df1,index_col=0)
df2 = pd.read_csv(path_df2,index_col=0)

print(f"Dataset 1: {df1.shape[0]} filas × {df1.shape[1]} columnas")
print(f"Dataset 2: {df2.shape[0]} filas × {df2.shape[1]} columnas")
display(df1.head())
display(df2.head())

Dataset 1: 499 filas × 22 columnas
Dataset 2: 20000 filas × 30 columnas


,Gender,Marital Status,Occupation,Educational Qualifications,Family size,Frequently used Medium,Frequently ordered Meal category,Perference,Restaurnat Rating,Delivery Rating,No. of orders placed,Delivery Time,Order Value,Ease and convenient,Self Cooking,Health Concern,Late Delivery,Poor Hygiene,Bad past experience,More Offers and Discount,Maximum wait time,Influence of rating
Age,,,,,,,,,,,,,,,,,,,,,,
20,Female,Single,Student,Post Graduate,4,Food delivery apps,Breakfast,Non Veg foods (Lunch / Dinner),1,1,150,45,1,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,30 minutes,Yes
24,Female,Single,Student,Graduate,3,Food delivery apps,Snacks,Non Veg foods (Lunch / Dinner),1,1,32,61,3,Strongly agree,Strongly agree,Strongly agree,Agree,Strongly agree,Strongly agree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,3,Food delivery apps,Lunch,Non Veg foods (Lunch / Dinner),3,5,193,19,1,Strongly agree,Disagree,Neutral,Neutral,Agree,Agree,Neutral,45 minutes,Yes
22,Female,Single,Student,Graduate,6,Food delivery apps,Snacks,Veg foods (Breakfast / Lunch / Dinner),2,4,60,44,1,Agree,Agree,Strongly agree,Neutral,Agree,Disagree,Strongly agree,30 minutes,Yes
22,Male,Single,Student,Post Graduate,4,Walk-in,Lunch,Non Veg foods (Lunch / Dinner),3,1,234,24,3,Agree,Agree,Strongly agree,Strongly agree,Agree,Strongly agree,Agree,30 minutes,Yes


,restaurant_id,food_item,order_time,delivery_time,delivery_distance,order_value,delivery_method,traffic_condition,weather_condition,delivery_delay,route_taken,customer_id,age,gender,location,order_history,customer_rating,preferred_cuisine,order_frequency,loyalty_program,food_temperature,food_freshness,packaging_quality,food_condition,customer_satisfaction,small_route,bike_friendly_route,route_type,route_efficiency,traffic_avoidance
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ORD000001,16,Taccos,2024-01-31,2024-01-31,2.17,42.21,Bike,Medium,Snowy,17.15,Route_4,CUST000001,32,Female,Hyderabad,46,4,American,Monthly,Yes,Hot,5,1,Fair,3,False,False,Bike-friendly,0.801291,Yes
ORD000002,30,Briyani rice,2024-10-16,2024-10-16,13.40,24.82,Car,High,Sunny,16.04,Route_2,CUST000002,59,Male,Lucknow,19,1,Asian,Weekly,No,Warm,3,2,Fair,5,False,False,Bike-friendly,0.645795,No
ORD000003,3,Pasta,2024-09-11,2024-09-11,10.74,37.25,Walk,High,Snowy,9.09,Route_3,CUST000003,20,Male,Chennai,10,5,Mexican,Monthly,Yes,Warm,4,5,Good,5,True,False,Bike-friendly,0.291193,No
ORD000004,93,Whole cake,2024-09-17,2024-09-17,6.29,49.88,Bike,High,Snowy,-2.11,Route_5,CUST000004,31,Male,Bangalore,6,4,American,Monthly,Yes,Cold,2,3,Poor,4,False,False,Car-only,0.133827,No
ORD000005,15,Sushi,2024-08-26,2024-08-26,2.94,8.53,Walk,Low,Sunny,12.20,Route_5,CUST000005,27,Other,Ahmedabad,24,3,Asian,Weekly,Yes,Cold,4,5,Good,1,False,True,Car-only,0.349233,No


#### **HOMOGENEIZACIÓN EN EL NOMBRADO DE COLUMNAS**

- En esta sección se estandarizan los nombres de las columnas para garantizar consistencia y facilitar su uso en las fases posteriores del análisis.
- Primero, se normalizan los nombres al formato `snake_case`, eliminando espacios y caracteres especiales.
- Posteriormente, se renombran aquellas variables con nomenclatura ambigua para mejorar su claridad semántica.
- Este proceso mejora la legibilidad del dataset y facilita su manipulación en el flujo de limpieza y análisis.

In [72]:
def normalizar_nombres_columnas(lista_columnas, mostrar_resumen=True):
    """
    Normaliza los nombres de columnas a formato snake_case estándar.

    Transformaciones aplicadas:
    - Eliminación de espacios al inicio y final
    - Conversión de CamelCase y PascalCase a snake_case
    - Sustitución de caracteres no alfanuméricos por guión bajo (_)
    - Conversión a minúsculas
    - Eliminación de guiones bajos duplicados
    - Eliminación de guiones bajos al inicio y final

    Parámetros
    ----------
    lista_columnas : list of str
        Lista de nombres de columnas originales.

    mostrar_resumen : bool, default=True
        Si True, muestra un resumen de la transformación realizada.

    Returns
    -------
    list of str
        Lista de nombres normalizados en formato snake_case.

    Ejemplo
    -------
    >>> normalizar_nombres_columnas(["Date&Time", "UserID", "App Name"])
    ['date_time', 'user_id', 'app_name']
    """

    nombres_normalizados = []

    for nombre in lista_columnas:

        # Eliminación de espacios extremos
        limpia = nombre.strip()

        # Conversión CamelCase / PascalCase → snake_case
        limpia = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', limpia)

        # Sustitución de caracteres no alfanuméricos por _
        limpia = re.sub(r'[^0-9a-zA-Z]+', '_', limpia)

        # Conversión a minúsculas
        limpia = limpia.lower()

        # Eliminación de guiones bajos duplicados
        limpia = re.sub(r'_+', '_', limpia)

        # Eliminación de guiones bajos en extremos
        limpia = limpia.strip('_')

        nombres_normalizados.append(limpia)

    if mostrar_resumen:
        print("Normalización de nombres de columnas completada.")
        print(f"Columnas procesadas: {len(lista_columnas)}")
        print("Transformaciones aplicadas:")
        for original, nuevo in zip(lista_columnas, nombres_normalizados):
            if original != nuevo:
                print(f"  {original} → {nuevo}")

    return nombres_normalizados

In [73]:
df2.columns = normalizar_nombres_columnas(df2.columns)

Normalización de nombres de columnas completada.
Columnas procesadas: 30
Transformaciones aplicadas:


In [74]:
df1.columns = normalizar_nombres_columnas(df1.columns)

Normalización de nombres de columnas completada.
Columnas procesadas: 22
Transformaciones aplicadas:
  Gender → gender
  Marital Status → marital_status
  Occupation → occupation
  Educational Qualifications → educational_qualifications
  Family size → family_size
  Frequently used Medium → frequently_used_medium
  Frequently ordered Meal category  → frequently_ordered_meal_category
  Perference → perference
  Restaurnat Rating → restaurnat_rating
  Delivery Rating → delivery_rating
  No. of orders placed → no_of_orders_placed
  Delivery Time → delivery_time
  Order Value → order_value
  Ease and convenient → ease_and_convenient
  Self Cooking → self_cooking
  Health Concern → health_concern
  Late Delivery → late_delivery
  Poor Hygiene → poor_hygiene
  Bad past experience → bad_past_experience
  More Offers and Discount → more_offers_and_discount
  Maximum wait time → maximum_wait_time
  Influence of rating → influence_of_rating


In [75]:
df2.columns

Index(['restaurant_id', 'food_item', 'order_time', 'delivery_time',
       'delivery_distance', 'order_value', 'delivery_method',
       'traffic_condition', 'weather_condition', 'delivery_delay',
       'route_taken', 'customer_id', 'age', 'gender', 'location',
       'order_history', 'customer_rating', 'preferred_cuisine',
       'order_frequency', 'loyalty_program', 'food_temperature',
       'food_freshness', 'packaging_quality', 'food_condition',
       'customer_satisfaction', 'small_route', 'bike_friendly_route',
       'route_type', 'route_efficiency', 'traffic_avoidance'],
      dtype='object')

In [76]:
df1.columns

Index(['gender', 'marital_status', 'occupation', 'educational_qualifications',
       'family_size', 'frequently_used_medium',
       'frequently_ordered_meal_category', 'perference', 'restaurnat_rating',
       'delivery_rating', 'no_of_orders_placed', 'delivery_time',
       'order_value', 'ease_and_convenient', 'self_cooking', 'health_concern',
       'late_delivery', 'poor_hygiene', 'bad_past_experience',
       'more_offers_and_discount', 'maximum_wait_time', 'influence_of_rating'],
      dtype='object')

#### **LIMPIEZA DE VARIABLES CATEGÓRICAS**

- En esta sección se estandariza el formato de las variables categóricas.
- Se eliminan espacios al inicio y al final de los valores y se aplica un formato homogéneo de capitalización.
- Este proceso garantiza la consistencia de las categorías y evita la duplicación de valores equivalentes con distinta representación.

In [77]:
def limpiar_categoricas(df, columnas=None):
    """
    Limpia variables categóricas eliminando espacios y unificando el formato del texto.

    Aplica .str.strip().str.title() para garantizar consistencia en las categorías.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame a procesar.

    columnas : list, default=None
        Columnas a limpiar. Si es None, se aplicará a todas las columnas categóricas.

    Devuelve
    --------
    pandas.DataFrame
        DataFrame con las variables categóricas limpias.
    """

    # Si no se especifican columnas, detectar todas las categóricas
    if columnas is None:
        columnas = df.select_dtypes(include="O").columns

    # Aplicar limpieza básica
    for col in columnas:
        df[col] = df[col].str.strip().str.title()

    return df

In [78]:
df1 = limpiar_categoricas(df1)
df1.head()

,gender,marital_status,occupation,educational_qualifications,family_size,frequently_used_medium,frequently_ordered_meal_category,perference,restaurnat_rating,delivery_rating,no_of_orders_placed,delivery_time,order_value,ease_and_convenient,self_cooking,health_concern,late_delivery,poor_hygiene,bad_past_experience,more_offers_and_discount,maximum_wait_time,influence_of_rating
Age,,,,,,,,,,,,,,,,,,,,,,
20,Female,Single,Student,Post Graduate,4,Food Delivery Apps,Breakfast,Non Veg Foods (Lunch / Dinner),1,1,150,45,1,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,30 Minutes,Yes
24,Female,Single,Student,Graduate,3,Food Delivery Apps,Snacks,Non Veg Foods (Lunch / Dinner),1,1,32,61,3,Strongly Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Strongly Agree,Strongly Agree,30 Minutes,Yes
22,Male,Single,Student,Post Graduate,3,Food Delivery Apps,Lunch,Non Veg Foods (Lunch / Dinner),3,5,193,19,1,Strongly Agree,Disagree,Neutral,Neutral,Agree,Agree,Neutral,45 Minutes,Yes
22,Female,Single,Student,Graduate,6,Food Delivery Apps,Snacks,Veg Foods (Breakfast / Lunch / Dinner),2,4,60,44,1,Agree,Agree,Strongly Agree,Neutral,Agree,Disagree,Strongly Agree,30 Minutes,Yes
22,Male,Single,Student,Post Graduate,4,Walk-In,Lunch,Non Veg Foods (Lunch / Dinner),3,1,234,24,3,Agree,Agree,Strongly Agree,Strongly Agree,Agree,Strongly Agree,Agree,30 Minutes,Yes


In [79]:
df2 = limpiar_categoricas(df2)
df2.head()

,restaurant_id,food_item,order_time,delivery_time,delivery_distance,order_value,delivery_method,traffic_condition,weather_condition,delivery_delay,route_taken,customer_id,age,gender,location,order_history,customer_rating,preferred_cuisine,order_frequency,loyalty_program,food_temperature,food_freshness,packaging_quality,food_condition,customer_satisfaction,small_route,bike_friendly_route,route_type,route_efficiency,traffic_avoidance
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ORD000001,16,Taccos,2024-01-31,2024-01-31,2.17,42.21,Bike,Medium,Snowy,17.15,Route_4,Cust000001,32,Female,Hyderabad,46,4,American,Monthly,Yes,Hot,5,1,Fair,3,False,False,Bike-Friendly,0.801291,Yes
ORD000002,30,Briyani Rice,2024-10-16,2024-10-16,13.40,24.82,Car,High,Sunny,16.04,Route_2,Cust000002,59,Male,Lucknow,19,1,Asian,Weekly,No,Warm,3,2,Fair,5,False,False,Bike-Friendly,0.645795,No
ORD000003,3,Pasta,2024-09-11,2024-09-11,10.74,37.25,Walk,High,Snowy,9.09,Route_3,Cust000003,20,Male,Chennai,10,5,Mexican,Monthly,Yes,Warm,4,5,Good,5,True,False,Bike-Friendly,0.291193,No
ORD000004,93,Whole Cake,2024-09-17,2024-09-17,6.29,49.88,Bike,High,Snowy,-2.11,Route_5,Cust000004,31,Male,Bangalore,6,4,American,Monthly,Yes,Cold,2,3,Poor,4,False,False,Car-Only,0.133827,No
ORD000005,15,Sushi,2024-08-26,2024-08-26,2.94,8.53,Walk,Low,Sunny,12.20,Route_5,Cust000005,27,Other,Ahmedabad,24,3,Asian,Weekly,Yes,Cold,4,5,Good,1,False,True,Car-Only,0.349233,No


In [80]:
#quiero ver si la columna delivery time y order time tiene los mismos valores. 
df2["delivery_time"].value_counts()



delivery_time
2024-09-29    90
2024-04-25    88
2024-09-21    86
2024-04-06    86
2024-10-23    84
              ..
2024-04-10    32
2024-07-09    32
2024-08-21    32
2024-01-06    32
2024-06-07    30
Name: count, Length: 347, dtype: int64

In [81]:
df2["order_time"].value_counts()    

order_time
2024-09-29    90
2024-04-25    88
2024-09-21    86
2024-04-06    86
2024-10-23    84
              ..
2024-04-10    32
2024-07-09    32
2024-08-21    32
2024-01-06    32
2024-06-07    30
Name: count, Length: 347, dtype: int64

#### **TRATAMIENTO DE VALORES NULOS**

- En esta sección se gestionan los valores nulos identificados durante el EDA con el objetivo de garantizar la consistencia y calidad del dataset.  
- Se eliminan los registros que presentan valores nulos en variables categóricas o numéricas.
- Este enfoque es apropiado dado el volumen de nulos a gestionar, hay en varias variables pero su % es muy pequeño lo que no afectará de ninguna forma nuestro análisis.

In [82]:
def tratamiento_nulos(df, mostrar_resumen = True):
    """
    Gestiona valores nulos eliminando filas con registros incompletos en variables
    categóricas y numéricas.

    Procedimiento:
    1) Identifica columnas categóricas (object/category) con nulos y elimina filas con nulos en ellas.
    2) Identifica columnas numéricas con nulos y elimina filas con nulos en ellas.
    3) Devuelve el DataFrame resultante (para actualizarlo por asignación).

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame de entrada.
    mostrar_resumen : bool, default=True
        Si True, imprime métricas del impacto del filtrado.

    Returns
    -------
    pandas.DataFrame
        DataFrame filtrado tras eliminar filas con nulos en columnas categóricas y numéricas.

    Nota
    ----
    Para “actualizar el dataframe”, reasigna:
    >>> df = tratamiento_nulos(df)
    """

    filas_iniciales = df.shape[0]

    # ---------- CATEGÓRICAS ----------
    cols_cat = df.select_dtypes(include=["object", "category"]).columns.tolist()
    cols_cat_nulos = [c for c in cols_cat if df[c].isna().any()]

    if cols_cat_nulos:
        filas_antes = df.shape[0]
        df = df.dropna(subset=cols_cat_nulos)
        filas_eliminadas_cat = filas_antes - df.shape[0]
    else:
        filas_eliminadas_cat = 0

    # ---------- NUMÉRICAS ----------
    cols_num = df.select_dtypes(include=["number"]).columns.tolist()
    cols_num_nulos = [c for c in cols_num if df[c].isna().any()]

    if cols_num_nulos:
        filas_antes = df.shape[0]
        df = df.dropna(subset=cols_num_nulos)
        filas_eliminadas_num = filas_antes - df.shape[0]
    else:
        filas_eliminadas_num = 0

    # ---------- RESUMEN ----------
    if mostrar_resumen:
        filas_finales = df.shape[0]
        print("Tratamiento de nulos completado")
        print(f"Filas iniciales: {filas_iniciales}")
        print(f"Filas eliminadas (categóricas): {filas_eliminadas_cat}")
        print(f"Filas eliminadas (numéricas): {filas_eliminadas_num}")
        print(f"Filas finales: {filas_finales}")
        print(f"Nulos restantes en el dataset: {df.isna().sum().sum()}")

        if cols_cat_nulos:
            print(f"Columnas categóricas con nulos consideradas: {cols_cat_nulos}")
        if cols_num_nulos:
            print(f"Columnas numéricas con nulos consideradas: {cols_num_nulos}")

    return df


In [83]:
#se manejan los nulos del primer dataset ya que en el segundo no hay nulos.
df1 = tratamiento_nulos(df1)

Tratamiento de nulos completado
Filas iniciales: 499
Filas eliminadas (categóricas): 12
Filas eliminadas (numéricas): 0
Filas finales: 487
Nulos restantes en el dataset: 0
Columnas categóricas con nulos consideradas: ['gender', 'occupation', 'frequently_used_medium', 'frequently_ordered_meal_category', 'perference', 'delivery_time', 'ease_and_convenient', 'health_concern', 'bad_past_experience', 'more_offers_and_discount', 'maximum_wait_time']


#### **PROCESAMIENTO DE VARIABLES TEMPORALES**

- En esta sección se convierte la variable temporal `date_time` al tipo datetime para garantizar su correcto tratamiento analítico.
- Esta transformación permite interpretar la variable como una dimensión temporal real, en lugar de texto, asegurando coherencia en ordenaciones y agregaciones.
- A partir de esta variable, se generan nuevas features temporales (`year`, `month`, `day`, `weekday`) que facilitan el análisis de patrones de consumo de la comida.
- Este procesamiento permite analizar día, hora, año y mes de cuando se realizo el pedido para poder determinar estacionalidad y comportamiento de los usuarios a lo largo del tiempo, mejorando la capacidad analítica del dataset.

In [84]:
def procesar_variable_temporal(
    df,
    col_fecha = "datetime",
    eliminar_original = False,
    mostrar_resumen = True
):
    """
    Convierte una columna a tipo datetime y genera variables temporales derivadas
    para facilitar el análisis temporal.

    Variables creadas:
    - year: año
    - month: mes (numérico)
    - month_name: nombre del mes
    - day: día del mes
    - weekday: nombre del día de la semana
    - hour: hora del día
    - date: fecha (sin hora)

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame de entrada.

    col_fecha : str, default="datetime"
        Nombre de la columna temporal a procesar.

    eliminar_original : bool, default=False
        Si True, elimina la columna original tras crear las variables derivadas.

    mostrar_resumen : bool, default=True
        Si True, muestra un resumen del proceso.

    Returns
    -------
    pandas.DataFrame
        DataFrame con la variable temporal convertida y nuevas features creadas.

    Ejemplo
    -------
    >>> df = procesar_variable_temporal(df, "date_time")
    """

    df = df.copy()

    # Validación
    if col_fecha not in df.columns:
        raise ValueError(f"La columna '{col_fecha}' no existe en el DataFrame")

    # Conversión a datetime
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")

    # Métricas
    nulos_generados = df[col_fecha].isna().sum()

    # Creación de variables derivadas
    df["year"] = df[col_fecha].dt.year
    df["month"] = df[col_fecha].dt.month
    df["month_name"] = df[col_fecha].dt.month_name()
    df["day"] = df[col_fecha].dt.day
    df["weekday"] = df[col_fecha].dt.day_name()
    
    # Eliminar original si se solicita
    if eliminar_original:
        df.drop(columns=[col_fecha], inplace=True)

    # Resumen
    if mostrar_resumen:
        print("Procesamiento de variable temporal completado")
        print(f"Columna procesada: {col_fecha}")
        print(f"Nulos generados tras conversión: {nulos_generados}")
        print("Variables creadas: year, month, month_name, day, weekday")

    return df


In [85]:
df2 = procesar_variable_temporal(df2, col_fecha="delivery_time")

Procesamiento de variable temporal completado
Columna procesada: delivery_time
Nulos generados tras conversión: 0
Variables creadas: year, month, month_name, day, weekday


In [86]:
df2.head()

,restaurant_id,food_item,order_time,delivery_time,delivery_distance,order_value,delivery_method,traffic_condition,weather_condition,delivery_delay,route_taken,customer_id,age,gender,location,order_history,customer_rating,preferred_cuisine,order_frequency,loyalty_program,food_temperature,food_freshness,packaging_quality,food_condition,customer_satisfaction,small_route,bike_friendly_route,route_type,route_efficiency,traffic_avoidance,year,month,month_name,day,weekday
order_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ORD000001,16,Taccos,2024-01-31,2024-01-31,2.17,42.21,Bike,Medium,Snowy,17.15,Route_4,Cust000001,32,Female,Hyderabad,46,4,American,Monthly,Yes,Hot,5,1,Fair,3,False,False,Bike-Friendly,0.801291,Yes,2024,1,January,31,Wednesday
ORD000002,30,Briyani Rice,2024-10-16,2024-10-16,13.40,24.82,Car,High,Sunny,16.04,Route_2,Cust000002,59,Male,Lucknow,19,1,Asian,Weekly,No,Warm,3,2,Fair,5,False,False,Bike-Friendly,0.645795,No,2024,10,October,16,Wednesday
ORD000003,3,Pasta,2024-09-11,2024-09-11,10.74,37.25,Walk,High,Snowy,9.09,Route_3,Cust000003,20,Male,Chennai,10,5,Mexican,Monthly,Yes,Warm,4,5,Good,5,True,False,Bike-Friendly,0.291193,No,2024,9,September,11,Wednesday
ORD000004,93,Whole Cake,2024-09-17,2024-09-17,6.29,49.88,Bike,High,Snowy,-2.11,Route_5,Cust000004,31,Male,Bangalore,6,4,American,Monthly,Yes,Cold,2,3,Poor,4,False,False,Car-Only,0.133827,No,2024,9,September,17,Tuesday
ORD000005,15,Sushi,2024-08-26,2024-08-26,2.94,8.53,Walk,Low,Sunny,12.20,Route_5,Cust000005,27,Other,Ahmedabad,24,3,Asian,Weekly,Yes,Cold,4,5,Good,1,False,True,Car-Only,0.349233,No,2024,8,August,26,Monday


#### **EXPORTACIÓN DEL DATASET LIMPIO**

- En esta sección se exportan los datasets tras aplicar los procesos de limpieza, tratamiento de nulos y transformación de variables.
- Los datasets se guardan en formato `.csv` dentro del directorio `data/procesada/`, siguiendo una estructura organizada del proyecto.
- Esta exportación garantiza la reutilización de los mismos sin necesidad de repetir las fases de preparación.

In [87]:
output_path_delivery_customer = "../Data/procesada/Customer online delivery dataset - Customer_data.csv"
output_path_delivery_service = "../Data/procesada/food_delivery_dataset.csv"

df1.to_csv(output_path_delivery_customer, index=False)
df2.to_csv(output_path_delivery_service, index=False)

print("Exportación completada correctamente:")
print(f"  - Customer data:  {output_path_delivery_customer}")
print(f"  - Service data:   {output_path_delivery_service}")

Exportación completada correctamente:
  - Customer data:  ../Data/procesada/Customer online delivery dataset - Customer_data.csv
  - Service data:   ../Data/procesada/food_delivery_dataset.csv
